# Pizza Customer & Product Marketing Analytics

### From sales behavior to marketing opportunities

This case study uses the Pizza Sales dataset to translate transaction-level sales behavior into **customer-behavior signals, product marketing opportunities, promotion timing, and merchandising recommendations**.

> **Important limitation:** the dataset has no customer ID, channel, campaign, discount, or acquisition-source fields. Therefore this project does **not** claim individual customer lifetime value, repeat-customer rate, campaign ROI, or true demographic segments. Segmentation is behavioral and order/product based.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../source_data/pizza_sales.csv")

df["order_date"] = pd.to_datetime(df["order_date"], dayfirst=True)
df["order_time"] = pd.to_datetime(df["order_time"], format="%H:%M:%S")
df["hour"] = df["order_time"].dt.hour
df["weekday"] = df["order_date"].dt.day_name()

def daypart(h):
    if 6 <= h < 11: return "Morning"
    if 11 <= h < 15: return "Lunch"
    if 15 <= h < 18: return "Afternoon"
    if 18 <= h < 23: return "Evening"
    return "Late Night"

df["daypart"] = df["hour"].map(daypart)

df.head()


## 1. Data Quality Check

In [ ]:
quality = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Duplicate rows", "Missing cells",
               "Unique orders", "Unique pizza products", "Start date", "End date"],
    "Value": [
        len(df), df.shape[1], df.duplicated().sum(), int(df.isna().sum().sum()),
        df["order_id"].nunique(), df["pizza_name"].nunique(),
        df["order_date"].min().date(), df["order_date"].max().date()
    ]
})
quality


## 2. Marketing KPI Baseline

In [ ]:
total_revenue = df["total_price"].sum()
total_pizzas = df["quantity"].sum()
total_orders = df["order_id"].nunique()
aov = total_revenue / total_orders
avg_pizzas_per_order = total_pizzas / total_orders

kpis = pd.Series({
    "Total revenue": total_revenue,
    "Total orders": total_orders,
    "Total pizzas sold": total_pizzas,
    "Average order value": aov,
    "Average pizzas per order": avg_pizzas_per_order
})
kpis


## 3. When Should Marketing Be Active?

In [ ]:
daypart_stats = (
    df.groupby("daypart")
      .agg(orders=("order_id","nunique"),
           revenue=("total_price","sum"),
           pizzas=("quantity","sum"))
      .sort_values("orders", ascending=False)
)
daypart_stats


In [ ]:
hourly = df.groupby("hour").agg(
    orders=("order_id","nunique"),
    revenue=("total_price","sum"),
    pizzas=("quantity","sum")
).sort_values("orders", ascending=False)

hourly.head(10)


### Marketing interpretation

Use high-demand periods for **conversion-focused messaging** and lower-demand periods for **targeted offers** rather than treating every hour equally.

The dataset shows a strong lunch/evening pattern. This supports time-windowed messaging such as lunch bundles and evening family/group offers, while avoiding unsupported claims about individual customer demographics.


## 4. Product Marketing Opportunities

In [ ]:
product = df.groupby("pizza_name").agg(
    revenue=("total_price","sum"),
    quantity=("quantity","sum"),
    orders=("order_id","nunique"),
    avg_unit_price=("unit_price","mean")
)

product["revenue_per_order"] = product["revenue"] / product["orders"]
product.sort_values("revenue", ascending=False).head(10)


In [ ]:
top_quantity = product.sort_values("quantity", ascending=False).head(10)
top_revenue = product.sort_values("revenue", ascending=False).head(10)

display(top_quantity[["quantity","orders","revenue"]])
display(top_revenue[["revenue","quantity","orders","avg_unit_price"]])


### Product strategy

Separate products into three practical marketing groups:

- **Volume drivers:** high quantity sold → suitable for reach and acquisition messaging.
- **Revenue drivers:** high revenue → suitable for hero-product placement and premium merchandising.
- **Long-tail products:** lower volume → candidates for testing, bundles, seasonal promotion, or menu rationalization.

This is a product-marketing framework, not a claim that low-selling products should automatically be removed.


## 5. Category & Size Positioning

In [ ]:
category = df.groupby("pizza_category").agg(
    revenue=("total_price","sum"),
    quantity=("quantity","sum"),
    orders=("order_id","nunique")
)
category["revenue_share"] = category["revenue"] / total_revenue
category.sort_values("revenue", ascending=False)


In [ ]:
size = df.groupby("pizza_size").agg(
    revenue=("total_price","sum"),
    quantity=("quantity","sum"),
    orders=("order_id","nunique")
)
size["revenue_share"] = size["revenue"] / total_revenue
size.sort_values("revenue", ascending=False)


## 6. Behavioral Order Segments

In [ ]:
order_level = df.groupby("order_id").agg(
    order_value=("total_price","sum"),
    pizzas=("quantity","sum"),
    distinct_products=("pizza_name_id","nunique")
)

order_level["order_value"].describe()


In [ ]:
# Practical behavioral buckets based on order value.
order_level["value_segment"] = pd.cut(
    order_level["order_value"],
    bins=[-np.inf, 20, 40, 70, np.inf],
    labels=["Entry", "Core", "High-Value", "Basket Builder"]
)

segment_summary = order_level.groupby("value_segment", observed=False).agg(
    orders=("order_value","size"),
    avg_order_value=("order_value","mean"),
    avg_pizzas=("pizzas","mean")
)
segment_summary["order_share"] = segment_summary["orders"] / len(order_level)
segment_summary


### Important

These are **order-value behavioral segments**, not customer segments, because the source data does not contain a customer identifier. They can support promotional hypotheses but should not be presented as a CRM/customer-lifetime segmentation model.


## 7. Weekday Promotion Planning

In [ ]:
weekday = df.groupby("weekday").agg(
    orders=("order_id","nunique"),
    revenue=("total_price","sum")
).reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])

weekday


### Promotion framework

**High-demand days:** protect conversion and average order value with bundles, add-ons, and hero-product merchandising.

**Lower-demand days:** test targeted incentives, limited-time bundles, or reminder campaigns.

The dataset alone cannot measure incremental lift, so these should be treated as **testable campaign hypotheses**, not proven campaign effects.


## 8. Marketing Recommendations

### Recommendation 1 — Time-windowed offers
Prioritize messaging around the strongest lunch and evening demand windows.

### Recommendation 2 — Product-led campaigns
Use high-volume pizzas for broad-reach creative and high-revenue pizzas for hero-product placement.

### Recommendation 3 — Basket-building
Use the order-level distribution and multi-product orders to design bundle and add-on tests.

### Recommendation 4 — Demand shaping
Use quieter periods for controlled promotional experiments instead of discounting during already-strong demand.

### Recommendation 5 — Measurement plan
For future campaigns, capture campaign ID, channel, customer ID, offer code, impressions, clicks, conversions, and discount amount so that incremental performance and ROI can be measured.


## 9. Final Takeaway

The original dataset is fundamentally a **sales transaction dataset**. The marketing value comes from translating those transactions into decisions about **when to communicate, which products to feature, how to structure offers, and what additional tracking is needed to prove campaign impact**.

This project therefore complements the portfolio's campaign-performance project rather than duplicating it.
